# Week 8 — Evaluation Expansion

**Deliverable:** `06_evaluation.ipynb` with `metrics_summary.csv`.

Steps:
1. Rebuild the same 12-month train / latest-month test split used throughout
   this project (same as `03_baseline_model.py` and `05_advanced_models.ipynb`).
2. Retrain Linear Regression, Decision Tree, Random Forest, and the tuned
   Gradient Boosting model (reusing the best hyperparameters found in Week 7's
   `gbm_hyperparameter_tuning_results.csv`), so we have predictions on the
   test set for all four models.
3. Compute metrics beyond R²: RMSE, MAE, R², **MAPE**, and **MdAPE** for each
   model, both overall and broken out by price band.
4. Summarize insights — e.g. which price bands each model handles well vs.
   poorly — and save everything to `metrics_summary.csv`.

This notebook re-loads the `Processed_CRMLSSold*.csv` files produced by
`03_baseline_model.py`'s preprocessing step, so it should be run from the
same project folder as the earlier weekly notebooks.

In [4]:
import pandas
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_s2core

# Same GBM library detection as 05_advanced_models.ipynb, so this notebook
# runs end-to-end regardless of which gradient boosting library is installed.
GBM_LIBRARY = None
try:
    from xgboost import XGBRegressor
    GBM_LIBRARY = 'xgboost'
except Exception as e:
    print(f'XGBoost unavailable ({type(e).__name__}: {e}); trying LightGBM...')
    try:
        from lightgbm import LGBMRegressor
        GBM_LIBRARY = 'lightgbm'
    except Exception as e2:
        print(f'LightGBM unavailable ({type(e2).__name__}: {e2}); '
              f'falling back to sklearn GradientBoostingRegressor.')
        from sklearn.ensemble import GradientBoostingRegressor
        GBM_LIBRARY = 'sklearn_gbr'

print(f'Using gradient boosting library: {GBM_LIBRARY}')

Using gradient boosting library: xgboost


## Step 1: Rebuild the same train/test split

Identical to Step 1 of `05_advanced_models.ipynb` — same 16-month window,
12-months-train / latest-month-test split, same `numeric_features` +
one-hot `zip_*` / `district_*` columns as `model_features`.

In [6]:
months = ['202502', '202503', '202504', '202505', '202506', '202507',
'202508', '202509', '202510', '202511', '202512', '202601', '202602',
'202603', '202604', '202605']

target = 'ClosePrice'

numeric_features = [
    'BedroomsTotal', 'BathroomsTotalInteger', 'LivingArea', 'LotSizeSquareFeet',
    'YearBuilt', 'GarageSpaces', 'ViewYN', 'WaterfrontYN', 'BasementYN',
    'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN',
    'BedBathRatio', 'PropertyAge',
]

zip_col = 'PostalCode'

test_month = months[-1]
train_months = months[-13:-1]

print(f'Training on months: {train_months}')
print(f'Testing on month: {test_month}')

train_dfs = [pandas.read_csv(f'Processed_CRMLSSold{m}.csv') for m in train_months]
train_df = pandas.concat(train_dfs, ignore_index=True)
test_df = pandas.read_csv(f'Processed_CRMLSSold{test_month}.csv')

train_df['DistrictName'] = train_df['DistrictName'].fillna('Unknown')
test_df['DistrictName'] = test_df['DistrictName'].fillna('Unknown')

train_df[zip_col] = train_df[zip_col].astype(str)
test_df[zip_col] = test_df[zip_col].astype(str)

combined = pandas.concat([train_df, test_df], keys=['train', 'test'])
combined = pandas.get_dummies(combined, columns=[zip_col, 'DistrictName'],
                               prefix=['zip', 'district'])
train_df = combined.xs('train')
test_df = combined.xs('test')

zip_dummy_cols = [c for c in combined.columns if c.startswith('zip_')]
district_dummy_cols = [c for c in combined.columns if c.startswith('district_')]
model_features = numeric_features + zip_dummy_cols + district_dummy_cols

X_train, y_train = train_df[model_features], train_df[target]
X_test, y_test = test_df[model_features], test_df[target]

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')

Training on months: ['202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604']
Testing on month: 202605
X_train: (98567, 2261)  X_test: (9146, 2261)


## Step 2: Get test-set predictions for all four models

`model_comparison_results.csv` (from Week 7) already has RMSE/MAE/R² for
Linear Regression, Decision Tree, Random Forest, and Gradient Boosting, but
MAPE/MdAPE and the price-band breakdown need the raw predictions, which
weren't persisted. So this step retrains all four on the exact split rebuilt
above (Linear Regression, Decision Tree, and Random Forest with the same
`random_state=42` configuration used in the earlier weekly notebooks), and
refits the boosting model using the **best hyperparameters already found**
in Week 7's `gbm_hyperparameter_tuning_results.csv` — no need to re-run the
full grid search.

In [8]:
models = {}

lr = LinearRegression()
lr.fit(X_train, y_train)
models['Linear Regression'] = lr

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
models['Decision Tree'] = dt

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = rf

gbm_label = {
    'xgboost': 'XGBoost',
    'lightgbm': 'LightGBM',
    'sklearn_gbr': 'Gradient Boosting (sklearn)',
}[GBM_LIBRARY]

try:
    tuning_df = pandas.read_csv('gbm_hyperparameter_tuning_results.csv')
    best_row = tuning_df.sort_values('rmse').iloc[0]
    best_params = dict(
        max_depth=int(best_row['max_depth']),
        learning_rate=float(best_row['learning_rate']),
        n_estimators=int(best_row['n_estimators']),
    )
    print(f'Reusing tuned params from Week 7: {best_params}')
except FileNotFoundError:
    print('gbm_hyperparameter_tuning_results.csv not found -- using reasonable defaults.')
    best_params = dict(max_depth=5, learning_rate=0.05, n_estimators=400)

if GBM_LIBRARY == 'xgboost':
    gbm = XGBRegressor(subsample=0.8, colsample_bytree=0.8, random_state=42,
                        n_jobs=-1, tree_method='hist', **best_params)
elif GBM_LIBRARY == 'lightgbm':
    gbm = LGBMRegressor(subsample=0.8, colsample_bytree=0.8, random_state=42,
                         n_jobs=-1, **best_params)
else:
    gbm = GradientBoostingRegressor(subsample=0.8, random_state=42, **best_params)

gbm.fit(X_train, y_train)
models[gbm_label] = gbm

print(f'Trained models: {list(models.keys())}')

Reusing tuned params from Week 7: {'max_depth': 7, 'learning_rate': 0.1, 'n_estimators': 600}
Trained models: ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost']


## Step 3: Extended metrics — RMSE, MAE, R², MAPE, MdAPE

MAPE (Mean Absolute Percentage Error) and MdAPE (Median Absolute Percentage
Error) express error relative to the actual sale price, which is easier to
compare across cheap and expensive homes than a dollar-denominated RMSE/MAE.
MdAPE is included alongside MAPE since MAPE can be pulled up by a handful of
badly-mispriced outlier listings; MdAPE is more robust to that.

In [10]:
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

def mdape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.median(np.abs((y_true - y_pred) / y_true)) * 100)

overall_rows = []
predictions = {}
for name, model in models.items():
    preds = model.predict(X_test)
    predictions[name] = preds
    overall_rows.append({
        'model': name,
        'price_band': 'Overall',
        'n': len(y_test),
        'rmse': np.sqrt(mean_squared_error(y_test, preds)),
        'mae': mean_absolute_error(y_test, preds),
        'r2': r2_score(y_test, preds),
        'mape': mape(y_test, preds),
        'mdape': mdape(y_test, preds),
    })

overall_df = pandas.DataFrame(overall_rows)
overall_df.round(3)

,model,price_band,n,rmse,mae,r2,mape,mdape
0,Linear Regression,Overall,9146,197240.757,134264.169,0.859,15.389,10.594
1,Decision Tree,Overall,9146,338905.282,214470.597,0.585,23.963,13.551
2,Random Forest,Overall,9146,270120.770,171381.967,0.736,19.981,11.376
3,XGBoost,Overall,9146,220220.524,153398.037,0.825,18.826,12.055


## Step 4: Price-band breakdown

Splits the test month into four price bands (quartiles of the actual
`ClosePrice`) and computes the same metrics within each band, per model.
This is what actually answers "which price bands does each model handle
well vs. poorly" rather than just an aggregate number.

In [12]:
band_labels = ['Low', 'Mid-Low', 'Mid-High', 'High']
price_bands = pandas.qcut(y_test, q=4, labels=band_labels)

band_rows = []
for name, preds in predictions.items():
    preds = pandas.Series(preds, index=y_test.index)
    for band in band_labels:
        mask = price_bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_test[mask], preds[mask]
        band_rows.append({
            'model': name,
            'price_band': band,
            'n': int(mask.sum()),
            'price_range': f'${yt.min():,.0f} - ${yt.max():,.0f}',
            'rmse': np.sqrt(mean_squared_error(yt, yp)),
            'mae': mean_absolute_error(yt, yp),
            'r2': r2_score(yt, yp) if mask.sum() > 1 else np.nan,
            'mape': mape(yt, yp),
            'mdape': mdape(yt, yp),
        })

band_df = pandas.DataFrame(band_rows)
band_df.round(3)

,model,price_band,n,price_range,rmse,mae,r2,mape,mdape
0,Linear Regression,Low,2287,"$26,000 - $612,500",137263.350,94012.956,-0.571,24.794,15.410
1,Linear Regression,Mid-Low,2299,"$613,000 - $860,000",135125.292,97536.165,-2.510,13.298,9.826
2,Linear Regression,Mid-High,2328,"$860,500 - $1,300,000",157592.832,116882.791,-0.410,11.042,8.298
3,Linear Regression,High,2232,"$1,301,000 - $2,717,000",308867.746,231466.701,0.296,12.441,10.581
4,Decision Tree,Low,2287,"$26,000 - $612,500",225784.494,129519.180,-3.250,36.242,13.404
5,Decision Tree,Mid-Low,2299,"$613,000 - $860,000",225392.050,136989.332,-8.767,18.589,10.811
6,Decision Tree,Mid-High,2328,"$860,500 - $1,300,000",287324.147,207347.502,-3.686,19.469,13.977
7,Decision Tree,High,2232,"$1,301,000 - $2,717,000",529129.437,388751.908,-1.067,21.605,16.264
8,Random Forest,Low,2287,"$26,000 - $612,500",188294.674,114089.574,-1.956,34.989,12.920
9,Random Forest,Mid-Low,2299,"$613,000 - $860,000",154346.650,97797.405,-3.580,13.320,8.223


## Step 5: Save `metrics_summary.csv`

Combines the overall (whole test month) metrics and the per-price-band
metrics into a single deliverable CSV, one row per model × band.

In [14]:
metrics_summary = pandas.concat([overall_df, band_df], ignore_index=True)
metrics_summary = metrics_summary[
    ['model', 'price_band', 'n', 'rmse', 'mae', 'r2', 'mape', 'mdape']
]
metrics_summary.to_csv('metrics_summary.csv', index=False)

print(f'Saved metrics_summary.csv ({len(metrics_summary)} rows)')
metrics_summary.round(3)

Saved metrics_summary.csv (20 rows)


,model,price_band,n,rmse,mae,r2,mape,mdape
0,Linear Regression,Overall,9146,197240.757,134264.169,0.859,15.389,10.594
1,Decision Tree,Overall,9146,338905.282,214470.597,0.585,23.963,13.551
2,Random Forest,Overall,9146,270120.770,171381.967,0.736,19.981,11.376
3,XGBoost,Overall,9146,220220.524,153398.037,0.825,18.826,12.055
4,Linear Regression,Low,2287,137263.350,94012.956,-0.571,24.794,15.410
5,Linear Regression,Mid-Low,2299,135125.292,97536.165,-2.510,13.298,9.826
6,Linear Regression,Mid-High,2328,157592.832,116882.791,-0.410,11.042,8.298
7,Linear Regression,High,2232,308867.746,231466.701,0.296,12.441,10.581
8,Decision Tree,Low,2287,225784.494,129519.180,-3.250,36.242,13.404
9,Decision Tree,Mid-Low,2299,225392.050,136989.332,-8.767,18.589,10.811


## Step 6: Insights — which price bands perform better?

Pulled directly from the numbers above (not hard-coded), so this stays
correct as the underlying models/data change. Reports the overall best
model by RMSE, the easiest/hardest price band on average across models, and
each model's individual best/worst band.

In [16]:
best_overall = overall_df.sort_values('rmse').iloc[0]

band_avg_mape = band_df.groupby('price_band')['mape'].mean().reindex(band_labels)
easiest_band = band_avg_mape.idxmin()
hardest_band = band_avg_mape.idxmax()

per_model_band_notes = []
for name in models.keys():
    sub = band_df[band_df['model'] == name].set_index('price_band').reindex(band_labels)
    best_band = sub['mape'].idxmin()
    worst_band = sub['mape'].idxmax()
    per_model_band_notes.append(
        f"  - {name}: best on '{best_band}' (MAPE={sub.loc[best_band,'mape']:.2f}%), "
        f"worst on '{worst_band}' (MAPE={sub.loc[worst_band,'mape']:.2f}%)"
    )

insights = f"""
Week 8 Evaluation Expansion -- Insights
=========================================
Overall best model (lowest test RMSE): {best_overall['model']}
  RMSE=${best_overall['rmse']:,.0f}  MAE=${best_overall['mae']:,.0f}  R2={best_overall['r2']:.3f}
  MAPE={best_overall['mape']:.2f}%  MdAPE={best_overall['mdape']:.2f}%

Price-band performance (averaged across all models, by MAPE):
{band_avg_mape.round(2).to_string()}

  Easiest price band overall: {easiest_band} (avg MAPE={band_avg_mape[easiest_band]:.2f}%)
  Hardest price band overall: {hardest_band} (avg MAPE={band_avg_mape[hardest_band]:.2f}%)

Per-model best/worst price band:
""" + "\n".join(per_model_band_notes) + "\n"

print(insights)
with open('week8_evaluation_notes.txt', 'w') as f:
    f.write(insights)
print('Saved week8_evaluation_notes.txt')


Week 8 Evaluation Expansion -- Insights
Overall best model (lowest test RMSE): Linear Regression
  RMSE=$197,241  MAE=$134,264  R2=0.859
  MAPE=15.39%  MdAPE=10.59%

Price-band performance (averaged across all models, by MAPE):
price_band
Low         32.44
Mid-Low     14.98
Mid-High    14.28
High        16.51

  Easiest price band overall: Mid-High (avg MAPE=14.28%)
  Hardest price band overall: Low (avg MAPE=32.44%)

Per-model best/worst price band:
  - Linear Regression: best on 'Mid-High' (MAPE=11.04%), worst on 'Low' (MAPE=24.79%)
  - Decision Tree: best on 'Mid-Low' (MAPE=18.59%), worst on 'Low' (MAPE=36.24%)
  - Random Forest: best on 'Mid-Low' (MAPE=13.32%), worst on 'Low' (MAPE=34.99%)
  - XGBoost: best on 'Mid-High' (MAPE=12.61%), worst on 'Low' (MAPE=33.74%)

Saved week8_evaluation_notes.txt


In [17]:
# Save the winning model and the exact feature list
import joblib

# Whichever model actually won on RMSE this run
best_model_name = best_overall['model']
best_model = models[best_model_name]

joblib.dump(best_model, 'model.pkl')

# CRITICAL: model_features order must match exactly at prediction time
joblib.dump(model_features, 'model_features.pkl')

# Dropdown options for the Streamlit app — strip the 'zip_'/'district_' prefix
zip_options = sorted(c.replace('zip_', '') for c in zip_dummy_cols)
district_options = sorted(c.replace('district_', '') for c in district_dummy_cols)
joblib.dump(zip_options, 'zip_options.pkl')
joblib.dump(district_options, 'district_options.pkl')

print(f"✅ Saved model.pkl using {best_model_name}")
print(f"✅ {len(model_features)} features, {len(zip_options)} ZIPs, {len(district_options)} districts")

✅ Saved model.pkl using Linear Regression
✅ 2261 features, 1936 ZIPs, 310 districts
